Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    _create_pipeline,
    StatsModelsEstimator,
    OptunaOptimizer,
    smote_objective,
    pure_smote_objective,
    GridSearchOptimizer,
    RANDOM_STATE,
)

seed = RANDOM_STATE

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'optuna-opt-mean-smote_default-model'
add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 400

save_model_and_metrics = True
metrics_file = 'metrics_modelling4_smote_mean_opt.xlsx'

## Optimization functions
Optuna will not be used further, except first try, it is enough to conduct grid search

In [5]:
def optimize_smote_optuna(
    target:str,
    estimator:object,
    estimator_params:dict,
    n_trials:int=n_trials,
    step_scoring_average:str=step_scoring_average,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    metrics_file:str=metrics_file,
    seed:int=seed,
):
    """
    Optimize the SMOTE parameters for a given estimator.

    Args:
        target: The target variable for the model.
        estimator: The machine learning estimator to be optimized.
        estimator_params: Parameters for the estimator.
        n_trials: The number of trials for optimization. Defaults to n_trials.
        model_postfix: A postfix to append to the model name. Defaults to model_postfix.
        add_smote: Whether to add SMOTE to the pipeline. Defaults to add_smote.
        is_smotenc: Whether to use SMOTENC for categorical features. Defaults to is_smotenc.
        smote_params: Parameters for SMOTE. Defaults to smote_params.
        scoring_metrics: Metrics for scoring the model. Defaults to scoring_metrics.
        save_model_and_metrics: Whether to save the model and metrics. Defaults to save_model_and_metrics.
    """
    
    smote_params = smote_params.copy()
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )

    opt = OptunaOptimizer(
        objective=partial(smote_objective, ml_pipe=ml_pipe),
        study_name="smote_study",
        direction="maximize",
        seed=seed,
    )

    opt.optimize(n_trials=n_trials)

    best_params = opt.study.best_params
    print('best_params')
    display(best_params)

    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params={**smote_params, **best_params},
        metrics_file=metrics_file,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# Logistic Regression

In [6]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:40:25,181] A new study created in memory with name: smote_study
[I 2025-04-16 00:40:25,256] Trial 0 finished with value: 0.808631916777597 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:40:25,324] Trial 1 finished with value: 0.8033759141203999 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:40:25,392] Trial 2 finished with value: 0.8082580539469134 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:40:25,459] Trial 3 finished with value: 0.8086807655106563 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.8086807655106563.
[I 2025-04-16 00:40:25,527] Trial 4 finished with value: 0.8069288014407415 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is 

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smote_optuna-opt-...
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.886574
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.799242
cv_test_accuracy_balanced_median,0.816667
cv_test_roc_auc_median,0.873065


In [7]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:40:26,189] A new study created in memory with name: smote_study
[I 2025-04-16 00:40:26,261] Trial 0 finished with value: 0.808631916777597 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:40:26,330] Trial 1 finished with value: 0.8033759141203999 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:40:26,399] Trial 2 finished with value: 0.8082580539469134 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.808631916777597.
[I 2025-04-16 00:40:26,468] Trial 3 finished with value: 0.8086807655106563 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.8086807655106563.
[I 2025-04-16 00:40:26,535] Trial 4 finished with value: 0.8069288014407415 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is 

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smote_optuna-opt-...
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.886574
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.799242
cv_test_accuracy_balanced_median,0.816667
cv_test_roc_auc_median,0.873065


Full grid search can find better solution, than Optuna with small number of iterations - like 40! It appears in some Random_seed

In [8]:
estimator = LogisticRegression
target = 'no_fragmentation'
estimator_params = {
    "fit_intercept": False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:40:27,179] A new study created in memory with name: smote_study
[I 2025-04-16 00:40:27,241] Trial 0 finished with value: 0.7953914760068441 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.7953914760068441.
[I 2025-04-16 00:40:27,299] Trial 1 finished with value: 0.7893690960867744 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.7953914760068441.
[I 2025-04-16 00:40:27,357] Trial 2 finished with value: 0.7927422008145589 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.7953914760068441.
[I 2025-04-16 00:40:27,416] Trial 3 finished with value: 0.7972483300592221 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 3 with value: 0.7972483300592221.
[I 2025-04-16 00:40:27,474] Trial 4 finished with value: 0.7927358172501057 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_smote_optu...
holdout_test_f1_macro,0.802991
holdout_test_accuracy_balanced,0.85
holdout_test_roc_auc,0.954545
holdout_test_f1,0.734694
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.826797
cv_test_accuracy_balanced_median,0.864469
cv_test_roc_auc_median,0.946886


# KNClassifier

In [9]:
estimator = KNeighborsClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:40:28,056] A new study created in memory with name: smote_study
[I 2025-04-16 00:40:28,153] Trial 0 finished with value: 0.839000057333054 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.839000057333054.
[I 2025-04-16 00:40:28,247] Trial 1 finished with value: 0.8366744957233221 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.839000057333054.
[I 2025-04-16 00:40:28,342] Trial 2 finished with value: 0.8275188914323743 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 0 with value: 0.839000057333054.
[I 2025-04-16 00:40:28,437] Trial 3 finished with value: 0.8339107074490688 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 0 with value: 0.839000057333054.
[I 2025-04-16 00:40:28,531] Trial 4 finished with value: 0.826598877826172 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best is tr

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "KNeighborsClassifier"


,0
target,splashing
model,KNeighborsClassifier_splashing_smote_optuna-op...
holdout_test_f1_macro,0.772036
holdout_test_accuracy_balanced,0.77662
holdout_test_roc_auc,0.862654
holdout_test_f1,0.829787
holdout_test_accuracy,0.786667
cv_test_f1_macro_median,0.844575
cv_test_accuracy_balanced_median,0.870743
cv_test_roc_auc_median,0.908669


In [10]:
estimator = KNeighborsClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:40:29,365] A new study created in memory with name: smote_study
[I 2025-04-16 00:40:29,452] Trial 0 finished with value: 0.8165889152726052 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8165889152726052.
[I 2025-04-16 00:40:29,536] Trial 1 finished with value: 0.8473500834496152 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8473500834496152.
[I 2025-04-16 00:40:29,618] Trial 2 finished with value: 0.8312963436962847 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8473500834496152.
[I 2025-04-16 00:40:29,701] Trial 3 finished with value: 0.8152523468202186 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8473500834496152.
[I 2025-04-16 00:40:29,784] Trial 4 finished with value: 0.8055981156379303 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "KNeighborsClassifier"


,0
target,no_fragmentation
model,KNeighborsClassifier_no_fragmentation_smote_op...
holdout_test_f1_macro,0.825397
holdout_test_accuracy_balanced,0.852273
holdout_test_roc_auc,0.943636
holdout_test_f1,0.755556
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.841642
cv_test_accuracy_balanced_median,0.897436
cv_test_roc_auc_median,0.951282


# SVC

In [11]:
estimator = SVC
target = 'splashing'
estimator_params = {
    'probability': True,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:40:30,537] A new study created in memory with name: smote_study
[I 2025-04-16 00:40:30,648] Trial 0 finished with value: 0.8617852500049572 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8617852500049572.
[I 2025-04-16 00:40:30,753] Trial 1 finished with value: 0.8644198887587763 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8644198887587763.
[I 2025-04-16 00:40:30,853] Trial 2 finished with value: 0.8550552362111962 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8644198887587763.
[I 2025-04-16 00:40:30,961] Trial 3 finished with value: 0.8529924934958182 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8644198887587763.
[I 2025-04-16 00:40:31,068] Trial 4 finished with value: 0.8716691814330063 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "SVC"


,0
target,splashing
model,SVC_splashing_smote_optuna-opt-mean-smote_defa...
holdout_test_f1_macro,0.810348
holdout_test_accuracy_balanced,0.80787
holdout_test_roc_auc,0.905864
holdout_test_f1,0.865979
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.844575
cv_test_accuracy_balanced_median,0.870743
cv_test_roc_auc_median,0.925697


In [12]:
estimator = SVC
target = 'no_fragmentation'
estimator_params = {
    'probability': True,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:40:31,983] A new study created in memory with name: smote_study
[I 2025-04-16 00:40:32,091] Trial 0 finished with value: 0.8317057142869242 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8317057142869242.
[I 2025-04-16 00:40:32,190] Trial 1 finished with value: 0.8346658965165396 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 1 with value: 0.8346658965165396.
[I 2025-04-16 00:40:32,285] Trial 2 finished with value: 0.8331099974208901 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 1 with value: 0.8346658965165396.
[I 2025-04-16 00:40:32,389] Trial 3 finished with value: 0.831839385795292 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 1 with value: 0.8346658965165396.
[I 2025-04-16 00:40:32,489] Trial 4 finished with value: 0.8150515241534569 and parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999}. Best 

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "SVC"


,0
target,no_fragmentation
model,SVC_no_fragmentation_smote_optuna-opt-mean-smo...
holdout_test_f1_macro,0.846814
holdout_test_accuracy_balanced,0.893182
holdout_test_roc_auc,0.949091
holdout_test_f1,0.791667
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.850704
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.954212


# CatBoost

In [13]:
estimator = CatBoostClassifier
target = 'splashing'
estimator_params = {
    'verbose': False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:40:33,322] A new study created in memory with name: smote_study
[I 2025-04-16 00:40:37,669] Trial 0 finished with value: 0.8868392980653509 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.98}. Best is trial 0 with value: 0.8868392980653509.
[I 2025-04-16 00:40:41,841] Trial 1 finished with value: 0.8825840948565951 and parameters: {'k_neighbors': 8, 'sampling_strategy': 0.84}. Best is trial 0 with value: 0.8868392980653509.
[I 2025-04-16 00:40:46,002] Trial 2 finished with value: 0.8968166188226805 and parameters: {'k_neighbors': 4, 'sampling_strategy': 0.6599999999999999}. Best is trial 2 with value: 0.8968166188226805.
[I 2025-04-16 00:40:50,294] Trial 3 finished with value: 0.8834258532725957 and parameters: {'k_neighbors': 3, 'sampling_strategy': 0.95}. Best is trial 2 with value: 0.8968166188226805.
[W 2025-04-16 00:40:51,690] Trial 4 failed with parameters: {'k_neighbors': 7, 'sampling_strategy': 0.8899999999999999} because of the following error: Keybo

KeyboardInterrupt: 

In [ ]:
estimator = CatBoostClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': False,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:37:00,192] A new study created in memory with name: smote_study
[I 2025-04-16 00:37:04,384] Trial 0 finished with value: 0.8555830265314074 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.8555830265314074.
[I 2025-04-16 00:37:08,903] Trial 1 finished with value: 0.8405977620993098 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 0 with value: 0.8555830265314074.
[I 2025-04-16 00:37:13,521] Trial 2 finished with value: 0.8470822269933509 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 0 with value: 0.8555830265314074.
[I 2025-04-16 00:37:17,897] Trial 3 finished with value: 0.8263156929352599 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 0 with value: 0.8555830265314074.
[I 2025-04-16 00:37:21,962] Trial 4 finished with value: 0.8375712706989421 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.8555830

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "CatBoostClassifier"


,0
target,no_fragmentation
model,CatBoostClassifier_no_fragmentation_smote_optu...
holdout_test_f1_macro,0.897727
holdout_test_accuracy_balanced,0.897727
holdout_test_roc_auc,0.985455
holdout_test_f1,0.85
holdout_test_accuracy,0.92
cv_test_f1_macro_median,0.898077
cv_test_accuracy_balanced_median,0.880037
cv_test_roc_auc_median,0.969231


# XGBoost

In [ ]:
estimator = XGBClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:37:48,785] A new study created in memory with name: smote_study
[I 2025-04-16 00:37:49,159] Trial 0 finished with value: 0.8814430012194674 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.8814430012194674.
[I 2025-04-16 00:37:49,503] Trial 1 finished with value: 0.8718906934704831 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 0 with value: 0.8814430012194674.
[I 2025-04-16 00:37:49,862] Trial 2 finished with value: 0.8710454743729211 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 0 with value: 0.8814430012194674.
[I 2025-04-16 00:37:50,166] Trial 3 finished with value: 0.8856688482197489 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 3 with value: 0.8856688482197489.
[I 2025-04-16 00:37:50,478] Trial 4 finished with value: 0.8783525777455836 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 3 with value: 0.8856688

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "XGBClassifier"


,0
target,splashing
model,XGBClassifier_splashing_smote_optuna-opt-mean-...
holdout_test_f1_macro,0.896019
holdout_test_accuracy_balanced,0.886574
holdout_test_roc_auc,0.931713
holdout_test_f1,0.929293
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.876935
cv_test_accuracy_balanced_median,0.876935
cv_test_roc_auc_median,0.947619


In [ ]:
estimator = XGBClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:37:52,774] A new study created in memory with name: smote_study
[I 2025-04-16 00:37:53,048] Trial 0 finished with value: 0.8324587276981351 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.8324587276981351.
[I 2025-04-16 00:37:53,314] Trial 1 finished with value: 0.8351289886541711 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.8351289886541711.
[I 2025-04-16 00:37:53,580] Trial 2 finished with value: 0.8340623129366472 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.8351289886541711.
[I 2025-04-16 00:37:53,855] Trial 3 finished with value: 0.8300010757269864 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 1 with value: 0.8351289886541711.
[I 2025-04-16 00:37:54,144] Trial 4 finished with value: 0.8228740520816257 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 1 with value: 0.8351289

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "XGBClassifier"


,0
target,no_fragmentation
model,XGBClassifier_no_fragmentation_smote_optuna-op...
holdout_test_f1_macro,0.885894
holdout_test_accuracy_balanced,0.904545
holdout_test_roc_auc,0.979091
holdout_test_f1,0.837209
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.864469
cv_test_roc_auc_median,0.965201


# AdaBoost

In [ ]:
estimator = AdaBoostClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:37:55,985] A new study created in memory with name: smote_study
[I 2025-04-16 00:37:56,257] Trial 0 finished with value: 0.8509611300826094 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.8509611300826094.
[I 2025-04-16 00:37:56,535] Trial 1 finished with value: 0.8090098898867842 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 0 with value: 0.8509611300826094.
[I 2025-04-16 00:37:56,810] Trial 2 finished with value: 0.827559027044411 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 0 with value: 0.8509611300826094.
[I 2025-04-16 00:37:57,075] Trial 3 finished with value: 0.8277416672101828 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 0 with value: 0.8509611300826094.
[I 2025-04-16 00:37:57,336] Trial 4 finished with value: 0.8397275802404781 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.85096113

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "AdaBoostClassifier"


,0
target,splashing
model,AdaBoostClassifier_splashing_smote_optuna-opt-...
holdout_test_f1_macro,0.777184
holdout_test_accuracy_balanced,0.770833
holdout_test_roc_auc,0.854552
holdout_test_f1,0.848485
holdout_test_accuracy,0.8
cv_test_f1_macro_median,0.784791
cv_test_accuracy_balanced_median,0.803406
cv_test_roc_auc_median,0.890867


In [ ]:
estimator = AdaBoostClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:37:59,239] A new study created in memory with name: smote_study
[I 2025-04-16 00:37:59,514] Trial 0 finished with value: 0.8094128362171009 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.8094128362171009.
[I 2025-04-16 00:37:59,788] Trial 1 finished with value: 0.8097273611358765 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.8097273611358765.
[I 2025-04-16 00:38:00,062] Trial 2 finished with value: 0.8123683290596094 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 2 with value: 0.8123683290596094.
[I 2025-04-16 00:38:00,323] Trial 3 finished with value: 0.7987463047876417 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 2 with value: 0.8123683290596094.
[I 2025-04-16 00:38:00,583] Trial 4 finished with value: 0.8420123745121898 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 4 with value: 0.8420123

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "AdaBoostClassifier"


,0
target,no_fragmentation
model,AdaBoostClassifier_no_fragmentation_smote_optu...
holdout_test_f1_macro,0.853293
holdout_test_accuracy_balanced,0.870455
holdout_test_roc_auc,0.949091
holdout_test_f1,0.790698
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.854396
cv_test_accuracy_balanced_median,0.854396
cv_test_roc_auc_median,0.925824


# Random Forest

In [ ]:
estimator = RandomForestClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:38:02,495] A new study created in memory with name: smote_study
[I 2025-04-16 00:38:02,952] Trial 0 finished with value: 0.8835234612558206 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.8835234612558206.
[I 2025-04-16 00:38:03,421] Trial 1 finished with value: 0.8738817968557401 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 0 with value: 0.8835234612558206.
[I 2025-04-16 00:38:03,905] Trial 2 finished with value: 0.8723554881469328 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 0 with value: 0.8835234612558206.
[I 2025-04-16 00:38:04,368] Trial 3 finished with value: 0.864486175913499 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 0 with value: 0.8835234612558206.
[I 2025-04-16 00:38:04,834] Trial 4 finished with value: 0.8578652419753017 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.88352346

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "RandomForestClassifier"


,0
target,splashing
model,RandomForestClassifier_splashing_smote_optuna-...
holdout_test_f1_macro,0.839525
holdout_test_accuracy_balanced,0.836806
holdout_test_roc_auc,0.931713
holdout_test_f1,0.886598
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.896201
cv_test_accuracy_balanced_median,0.891641
cv_test_roc_auc_median,0.94582


In [ ]:
estimator = RandomForestClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:38:08,046] A new study created in memory with name: smote_study
[I 2025-04-16 00:38:08,502] Trial 0 finished with value: 0.8296683067641274 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.8296683067641274.
[I 2025-04-16 00:38:08,981] Trial 1 finished with value: 0.8401925021163178 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.8401925021163178.
[I 2025-04-16 00:38:09,465] Trial 2 finished with value: 0.8342802877691692 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.8401925021163178.
[I 2025-04-16 00:38:09,932] Trial 3 finished with value: 0.8180915574394719 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 1 with value: 0.8401925021163178.
[I 2025-04-16 00:38:10,388] Trial 4 finished with value: 0.8062006024358022 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 1 with value: 0.8401925

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "RandomForestClassifier"


,0
target,no_fragmentation
model,RandomForestClassifier_no_fragmentation_smote_...
holdout_test_f1_macro,0.885894
holdout_test_accuracy_balanced,0.904545
holdout_test_roc_auc,0.985
holdout_test_f1,0.837209
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.881326
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.949634


# LightGBM

In [ ]:
estimator = LGBMClassifier
target = 'splashing'
estimator_params = {
    'verbose': -1,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-16 00:38:13,578] A new study created in memory with name: smote_study
[I 2025-04-16 00:38:14,736] Trial 0 finished with value: 0.8673790814566678 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.8673790814566678.
[I 2025-04-16 00:38:16,151] Trial 1 finished with value: 0.8758737419163747 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.8758737419163747.
[I 2025-04-16 00:38:17,502] Trial 2 finished with value: 0.8797970114916229 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 2 with value: 0.8797970114916229.
[I 2025-04-16 00:38:18,767] Trial 3 finished with value: 0.8641094462158291 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 2 with value: 0.8797970114916229.
[I 2025-04-16 00:38:20,008] Trial 4 finished with value: 0.8701966126202139 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 2 with value: 0.8797970

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LGBMClassifier"


,0
target,splashing
model,LGBMClassifier_splashing_smote_optuna-opt-mean...
holdout_test_f1_macro,0.86631
holdout_test_accuracy_balanced,0.857639
holdout_test_roc_auc,0.946759
holdout_test_f1,0.909091
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.898584
cv_test_accuracy_balanced_median,0.903251
cv_test_roc_auc_median,0.94582


In [ ]:
estimator = LGBMClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': -1,
}

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-16 00:38:28,870] A new study created in memory with name: smote_study
[I 2025-04-16 00:38:30,221] Trial 0 finished with value: 0.8223909832900251 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.8223909832900251.
[I 2025-04-16 00:38:31,952] Trial 1 finished with value: 0.8269598185455334 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.8269598185455334.
[I 2025-04-16 00:38:33,658] Trial 2 finished with value: 0.8303021436806487 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 2 with value: 0.8303021436806487.
[I 2025-04-16 00:38:35,078] Trial 3 finished with value: 0.837382919552037 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 3 with value: 0.837382919552037.
[I 2025-04-16 00:38:36,498] Trial 4 finished with value: 0.8190243682922306 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 3 with value: 0.837382919

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LGBMClassifier"


,0
target,no_fragmentation
model,LGBMClassifier_no_fragmentation_smote_optuna-o...
holdout_test_f1_macro,0.931818
holdout_test_accuracy_balanced,0.931818
holdout_test_roc_auc,0.991818
holdout_test_f1,0.9
holdout_test_accuracy,0.946667
cv_test_f1_macro_median,0.90293
cv_test_accuracy_balanced_median,0.887179
cv_test_roc_auc_median,0.967521
